# AI Ignite at York - Forge Track
## Forge Workshop 01 - The Scoreboard (evaluation notebook)

**Block 04 - Sheheryar - 7:03 to 7:22**

> *Opinions end where the numbers start, and accuracy on its own will lie to you on data this imbalanced.*

This notebook picks up **every** predictions CSV in `outputs/predictions/` automatically. Whatever models are there get scored, and if one is missing it simply isn't on the scoreboard.

Every model is scored on **the same lines of MELD's official test split**. The LLM runs cover the first 100 test lines (live LLM calls are capped at 100), so the scoreboard compares everyone on those 100. Models that ran on all 2,610 test lines also get a full-test-set table at the end.

Run it top to bottom. All the logic lives in `eval.py` next to this notebook, so every cell here stays short.

## Step 1 - Setup
Finds the workshop repo (in Colab it clones it first) and loads the evaluation code.

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/sagarh12/forge-workshop.git"

def find_root(start):
    for folder in [start, *start.parents]:
        if (folder / "forge_workshop_base.ipynb").exists():
            return folder
    return None

ROOT = find_root(Path.cwd().resolve())
if ROOT is None and "google.colab" in sys.modules:
    if not Path("forge-workshop").exists():
        subprocess.check_call(["git", "clone", "-q", REPO_URL])
    ROOT = Path("forge-workshop").resolve()
assert ROOT is not None, "Open this notebook from inside the forge-workshop repo."
os.chdir(ROOT)

sys.path.insert(0, str(ROOT / "notebooks" / "sheheryar"))
import eval as ev
importlib.reload(ev)  # picks up edits to eval.py without restarting

PRED_DIR = Path(os.environ.get("FORGE_PREDICTIONS_DIR", ROOT / "outputs" / "predictions"))
REPORTS_DIR = ROOT / "outputs" / "reports"
FIGURES_DIR = ROOT / "outputs" / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Repo:              ", ROOT)
print("Predictions folder:", PRED_DIR)

## Step 2 - Load every predictions CSV
No filenames are typed anywhere. The notebook takes whatever `*.csv` files are in the folder.

In [ ]:
frames = ev.load_predictions(PRED_DIR)
dummy_files = ev.find_dummy_files(PRED_DIR)

print(f"Found {len(frames)} predictions file(s):")
for name, df in frames.items():
    flag = "   <-- DUMMY DATA, not a real result" if name in dummy_files else ""
    print(f"  {name:<36} {len(df):>5} rows{flag}")

missing = [m for m in ev.MODEL_NAMES if f"{m}_predictions.csv" not in frames]
if missing:
    print("\nNot here (the notebook carries on without them):", ", ".join(missing))

## Step 3 - Alignment check (always first)
Before scoring anything, we prove every model was tested on **the same lines, in the same order**: the official MELD test split. Every file holds all 2,610 test lines, except the three LLM runs (zero-shot, few-shot, balanced few-shot), which may hold **exactly the first 100**. Everyone is then scored on the lines all files share.

If one model was scored on different rows (for example after a random re-split or a shuffle), its numbers can't be compared with the others. The chart would still look clean, so nothing would *visibly* break. That makes this the most dangerous bug in the workshop, so the check stops the notebook with a loud error naming the file and the first bad row.

In [ ]:
official_test = ev.load_official_test(ROOT)   # None until data/processed/meld_clean.csv is committed
alignment = ev.check_alignment(frames, official_test, dummy_files)
ev.print_alignment(alignment)
frames = alignment.frames   # from here on, only files that passed are scored

## Step 4 - How we are judging this (plain English)

- **Precision:** when the model says "angry", how often is it right? If it calls 10 lines angry and 7 really are, its precision for anger is 70%.
- **Recall:** of all the lines that really are angry, how many did the model catch? If there are 20 angry lines and it finds 7 of them, its recall for anger is 35%.
- **F1:** one score that is only high when precision *and* recall are both high, so 70% precision with 35% recall gives an F1 of about 47%, not the 52% a simple average would suggest.
- **Macro F1:** work out F1 for each of the seven emotions separately and average them, so *fear* (50 lines) counts exactly as much as *neutral* (over 1,000 lines).
- **Accuracy:** the share of all lines labelled correctly, which sounds fair until one emotion makes up half the data...

## Step 5 - Why accuracy lies
MELD is about **48% neutral**. So here is a "model" that never reads the text and just answers *neutral* every time.

In [ ]:
results = ev.evaluate_all(frames, dummy_files)
y_true = next(iter(frames.values()))["true_label"].tolist()
baseline = ev.majority_baseline(y_true)
majority = baseline.model_name.split("_", 1)[1]

print(f"Always answering '{majority}':  accuracy {ev.pct(baseline.accuracy)}   macro F1 {ev.pct(baseline.macro_f1)}\n")

beaten = [r for r in results if r.accuracy <= baseline.accuracy]
if beaten:
    names = ", ".join(f"{r.model_name} ({ev.pct(r.accuracy)})" for r in beaten)
    print(f"On accuracy, the do-nothing model beats or ties: {names}")
else:
    closest = min(results, key=lambda r: r.accuracy - baseline.accuracy)
    gap = (closest.accuracy - baseline.accuracy) * 100
    print(f"On accuracy, a model that never reads a single line still scores {ev.pct(baseline.accuracy)}. "
          f"The weakest real model on accuracy ({closest.model_name}) is {gap:.1f} points above it.")
print(f"But its macro F1 is {ev.pct(baseline.macro_f1)}: it gets 0% of the other six emotions right.\n")

print(f"{'':<20} {'Accuracy':>9} {'Macro F1':>9}")
for r in sorted([*results, baseline], key=lambda r: r.accuracy, reverse=True):
    print(f"{r.model_name:<20} {ev.pct(r.accuracy):>9} {ev.pct(r.macro_f1):>9}")

## Step 6 - Every model, in detail
For each model, here are the classification report (precision, recall and F1 per emotion) and a **confusion matrix**.

**How to read a confusion matrix:** each row is what the line *really* was, and each column is what the model *said*. The diagonal is where it got things right. Each row adds up to 100%, so the biggest number off the diagonal is the model's favourite mistake.

Reports are saved to `outputs/reports/`, and the figures to `outputs/figures/`.

In [ ]:
longer = ev.evaluate_longer(alignment, dummy_files)   # models that also ran on more lines (Step 7b)
removed = ev.remove_stale_outputs(REPORTS_DIR, FIGURES_DIR, [r.model_name for r in results],
                                  [r.model_name for r in longer])
if removed:
    print("Removed old outputs of models no longer in the folder:", ", ".join(removed), "\n")

for r in results:
    print("=" * 72)
    print(f"{r.title}    on {r.n_rows:,} test lines    accuracy {ev.pct(r.accuracy)}    macro F1 {ev.pct(r.macro_f1)}")
    print("=" * 72)
    print(r.report_text)
    ev.save_report(r, REPORTS_DIR)
    ev.plot_confusion_matrix(r, FIGURES_DIR / f"{r.model_name}_confusion_matrix.png")
    plt.show()
    t, p, share, n = ev.top_confusions(r, 1, include_neutral=False)[0]
    print(f"Most confused pair of emotions: {share:.0%} of real '{t}' lines were called '{p}' ({ev.lines(n)}).")
    t, p, share, n = ev.top_confusions(r, 1)[0]
    print(f"Biggest single mistake overall: {share:.0%} of real '{t}' lines were called '{p}' ({ev.lines(n)}).\n")

### Your turn
Pick a model by changing `MY_MODEL` below to one of `bilstm`, `bert_finetuned`, `zero_shot`, `few_shot` or `balanced_few_shot`. Look at its confusion matrix above: **which two emotions does it mix up most?** Say it out loud, then run the cell to check.

In [ ]:
MY_MODEL = "bert_finetuned"

chosen = next((r for r in results if r.model_name == MY_MODEL), results[0])
if chosen.model_name != MY_MODEL:
    print(f"{MY_MODEL} isn't in the folder, showing {chosen.model_name} instead.")
for t, p, share, n in ev.top_confusions(chosen, 3, include_neutral=False):
    print(f"{chosen.model_name}: {share:.0%} of real '{t}' lines were called '{p}' ({ev.lines(n)})")

## Step 7 - The scoreboard
Ranked by **macro F1**, the honest metric. The last column shows where each model would have ranked on accuracy alone. Wherever the two ranks disagree, accuracy was lying.

In [ ]:
board = ev.scoreboard(results)
board.to_csv(REPORTS_DIR / "scoreboard.csv", index=False)
print(f"Scored on {ev.describe_rows(alignment)}\n")
print(ev.format_scoreboard(board, baseline))

moved = board[board["rank"] != board["accuracy_rank"]]
if len(moved):
    print("\nWhere accuracy and macro F1 disagree:")
    for row in moved.itertuples():
        print(f"  {row.model:<20} #{row.accuracy_rank} on accuracy  ->  #{row.rank} on macro F1")

# A few points' lead can be luck: in the first 100 test lines there are only 4 fear and 2 disgust lines,
# so one right or wrong answer on a rare emotion swings macro F1 a lot. The bootstrap checks that.
if len(results) >= 2:
    ranked = sorted(results, key=lambda r: (r.macro_f1, r.accuracy), reverse=True)
    print("\n" + ev.describe_gap(ev.macro_f1_gap(ranked[0], ranked[1])))

### Step 7b - The trained models on the full test set
BERT and the BiLSTM are scored on all 2,610 test lines, not just the first 100. Here are those numbers on their own. Compare them with each other, **not** with the table above: different lines, different test.

In [ ]:
if longer:
    full_board = ev.scoreboard(longer)
    full_board.to_csv(REPORTS_DIR / f"scoreboard{ev.FULL_SUFFIX}.csv", index=False)
    print(f"Scored on all {longer[0].n_rows:,} lines of their files\n")
    print(ev.format_scoreboard(full_board, ev.majority_baseline(longer[0].y_true)))
    for r in longer:
        ev.save_report(r, REPORTS_DIR, ev.FULL_SUFFIX)
        plt.close(ev.plot_confusion_matrix(r, FIGURES_DIR / f"{r.model_name}_confusion_matrix{ev.FULL_SUFFIX}.png"))
    if len(longer) >= 2:
        print("\n" + ev.describe_gap(ev.macro_f1_gap(longer[0], longer[1]) if longer[0].macro_f1 >= longer[1].macro_f1
                                     else ev.macro_f1_gap(longer[1], longer[0])))
else:
    print("Every file covers the same lines, so the scoreboard above is the whole story.")

## Step 8 - The master comparison
Every approach, side by side, on identical metrics. Light bars show accuracy and dark bars show macro F1. The hatched bars on the right are the do-nothing baseline.

The leader only gets a **WINNER** label if its lead is bigger than chance. Otherwise the chart says it's too close to call.

In [ ]:
ev.plot_master_comparison(results, baseline, FIGURES_DIR / "master_comparison.png",
                          n_rows=alignment.n_rows, used_official_test=alignment.used_official_test,
                          test_rows=alignment.test_rows)
plt.show()

if dummy_files:
    print(f"WARNING: {len(dummy_files)} file(s) are DUMMY data: {', '.join(sorted(dummy_files))}")

---
## How prediction files arrive
Drop a CSV into `outputs/predictions/` (saved with `save_predictions()`, same filename) and re-run everything. Nothing in this notebook needs editing.
- Every file is checked against the **official test split** in `data/processed/meld_clean.csv`: the same lines, in the same order. Only the three LLM runs may stop early, and then only at **exactly the first 100** test lines. BERT and the BiLSTM must cover all 2,610. Nothing may skip, shuffle or re-split.
- The scoreboard uses the lines every file has. Today that's the first 100 (the LLM runs). Trained models that cover all 2,610 lines also appear in Step 7b.
- Dummy files from `make_dummy_predictions.py` are recognised automatically and labelled DUMMY. Leftover dummies never outvote real files.
- **Before the live run, delete any dummy CSVs that are still in the folder.** A model that hasn't been delivered should simply be missing from the scoreboard.
- Before committing a predictions CSV, run `python tests/test_alignment.py`.

*AI Ignite at York - Forge Track - Workshop 01 - November 2026*